# `parse_question` — Brique 2 : parser une question utilisateur en JSON

Référence : chapitre **« Understanding the Question Before Searching »** (4ème onglet du Google Doc Faseya).

**Idée centrale** : une question utilisateur n'est pas un seul input. C'est la matière première pour DEUX préparations distinctes :

| Préparation | Pour quoi | Contenu |
|---|---|---|
| `RetrievalQuery`  | la recherche dans les documents (brique 3) | `main_query`, `anchor_keywords`, `structural_hints` |
| `GenerationBrief` | la formulation de la réponse par le LLM (brique 4) | `original_question`, `format_constraint`, `disambiguation` |

`parse_question(question)` retourne un `ParsedQuestion` JSON-sérialisable qui contient les deux. Approche : **regex / heuristique uniquement, pas de LLM** (règle CLAUDE.md du repo : LLM réservé à translation / summarization / Excel SQL agent).

## Champs du JSON

| Champ | Type | Exemple |
|---|---|---|
| `original_question`  | str       | `"Le plafond, page 3, pas la franchise."` |
| `main_query`         | str       | `"Le plafond, page 3, pas la franchise."` (cleaned) |
| `structural_hints`   | object    | `{ page_hint: 3, section_hint: …, layout_hint: …, document_hint: … }` |
| `anchor_keywords`    | list[str] | `["L131-1", "GDPR", "RC-2024"]` |
| `format_constraint`  | str?      | `"ISO 8601 date (YYYY-MM-DD)"` |
| `disambiguation`     | object    | `{ distractors: ["franchise"], instruction: "..." }` |
| `intent`             | str       | `"extract"` / `"compare"` / `"aggregate"` / `"conditional"` / `"yes_no"` |
| `raw_signals`        | list[str] | traçabilité de tous les patterns matchés |

## Sortie consommée par la brique 3 Retrieval

Les `structural_hints` matchent directement les colonnes du `page_df` produit par `parse_pdf` (`page_num`, `has_vector_table`, `has_full_page_image`). Le retrieval filtrera d'abord par `page_hint`, puis par `layout_hint`, puis cherchera dans le `line_df` filtré.

In [ ]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ../../../..

In [ ]:
import json
from dataclasses import asdict
from docpipeline.question.question_parsing import parse_question

# Banque de questions exemples (à remplacer par celles de Kezhan)
QUESTIONS = [
    "Quelle est la date d'effet du contrat ? C'est en page 1, format YYYY-MM-DD.",
    "What's the limit per claim, not the deductible, in this contract?",
    "Look in the exclusions section for the flooding clause.",
    "Does article L131-1 of the insurance code apply here?",
    "Compare the indemnification and liability caps in the latest version.",
    "List all the obligations of the seller, in bullet list.",
    "Find the policy number — it's usually in the header, format JSON.",
    "Combien coûte l'assurance ?",
]

for q in QUESTIONS:
    parsed = parse_question(q)
    print(f"Q: {q}")
    print(json.dumps(asdict(parsed), indent=2, ensure_ascii=False))
    print('-' * 80)